In [1]:
import keras
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
DATA_DIR = "mu3e_trigger_data"
SIGNAL_DATA_FILE = f"{DATA_DIR}/run42_sig_positions.npy"
BACKGROUND_DATA_FILE = f"{DATA_DIR}/run42_bg_positions.npy"

max_barrel_radius = 86.3
max_endcap_distance = 372.6

In [8]:
signal_data = np.load(SIGNAL_DATA_FILE)
background_data = np.load(BACKGROUND_DATA_FILE)
from src.utils import normalize_data, cartesian_to_cylindrical
normed_signal_data = normalize_data(signal_data, type="minmax", feature_range=(0, 1), padding_value=-1)
normed_background_data = normalize_data(background_data, type="minmax", feature_range=(0, 1), padding_value=-1)

signal_data_cylindrical = cartesian_to_cylindrical(signal_data)
background_data_cylindrical = cartesian_to_cylindrical(background_data)

normed_signal_data_cylindrical = normalize_data(signal_data_cylindrical, type="minmax", feature_range=(0, 1), padding_value=-1)
normed_background_data_cylindrical = normalize_data(background_data_cylindrical, type="minmax", feature_range=(0, 1), padding_value=-1)

In [ ]:
zero_padded_bg = change_padding_value(background_data, -1, 0)
zero_padded_sig = change_padding_value(signal_data, -1, 0)

In [ ]:
flattened_bg = zero_padded_bg.reshape(zero_padded_bg.shape[0], -1)
flattened_sig = zero_padded_sig.reshape(zero_padded_sig.shape[0], -1)

In [ ]:
from src.model import AutoEncoder, VarAutoEncoder

autoencoder = VarAutoEncoder(
    input_size=flattened_bg.shape[1],
    latent_dim=32,
    num_layers=6,
)


In [ ]:
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
)
autoencoder.summary()

In [ ]:
from sklearn.model_selection import train_test_split
bg_train, bg_test = train_test_split(flattened_bg, test_size=0.2, random_state=42)

In [ ]:
autoencoder.fit(bg_train, epochs = 20, batch_size=512)

In [ ]:
bg_test_mean, bg_test_log_var, bg_test_z  = autoencoder.encoder.predict(bg_test)
sig_mean, sig_log_var, sig_z = autoencoder.encoder.predict(flattened_sig)

In [ ]:
bg_test_diff = bg_test - autoencoder.predict(bg_test)
signal_diff = flattened_sig - autoencoder.predict(flattened_sig)
bg_train_diff = bg_train - autoencoder.predict(bg_train)

In [ ]:
from src.evaluation import plot_latent_variable_distributions
# Example usage
plot_latent_variable_distributions(bg_test_log_var, sig_log_var)

In [ ]:

bg_ad_score = np.linalg.norm(bg_train_diff, axis=1)
signal_ad_score = np.linalg.norm(signal_diff, axis=1)

plot_latent_variable_distributions(bg_ad_score, signal_ad_score)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bg_ad_score = np.linalg.norm(bg_test_diff, axis=1)
bg_train_ad_score = np.linalg.norm(bg_train_diff, axis=1)
bins = np.linspace(
    min(np.min(bg_ad_score), np.min(bg_train_ad_score)),
    max(np.max(bg_ad_score), np.max(bg_train_ad_score)),
    30,
)
ax.hist(bg_ad_score, bins=bins, alpha=0.5, label="Test", color="blue", density=True)
ax.hist(bg_train_ad_score, bins=bins, alpha=0.5, label="Training", color="red", density=True)
ax.set_xlabel("Reconstruction Error")
ax.set_ylabel("Count")
ax.set_title("Reconstruction Error Distribution")
ax.legend()

In [ ]:
seq_length_sig = (signal_data != 0).all(axis = -1).sum(axis=1)
seq_length_bg = (background_data != 0).all(axis = -1).sum(axis=1)
bg_ad_score = np.linalg.norm(bg_test_diff, axis=1)
signal_ad_score = np.linalg.norm(signal_diff, axis=1)
bg_ad_score = np.max(bg_test_log_var, axis=1)
signal_ad_score = np.max(sig_log_var, axis=1)


In [ ]:
from sklearn.metrics import roc_curve, auc
ae_fpr, ae_tpr, ae_thresholds = roc_curve(
    np.concatenate([np.zeros(len(bg_test)), np.ones(len(flattened_sig))]),
    np.concatenate([bg_ad_score, signal_ad_score])
)
ae_roc_auc = auc(ae_fpr, ae_tpr)

seq_length_fpr, seq_length_tpr, seq_length_thresholds = roc_curve(
    np.concatenate([np.zeros(len(seq_length_bg)), np.ones(len(seq_length_sig))]),
    np.concatenate([seq_length_bg, seq_length_sig])
)
seq_length_roc_auc = auc(seq_length_fpr, seq_length_tpr)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ae_fpr, ae_tpr, color='blue', label=f'Autoencoder ROC curve (area = {ae_roc_auc:.2f})')
ax.plot(seq_length_fpr, seq_length_tpr, color='red', label=f'Sequence Length ROC curve (area = {seq_length_roc_auc:.2f})')
ax.plot([0, 1], [0, 1], color='black', linestyle  ='--', label='Random Guessing')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves for Autoencoder and Sequence Length')
ax.legend()
plt.show()